In [0]:
import os
import sys
import yaml
import argparse
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql import SparkSession

# 1. Parse DABs injected parameters (with Notebook fallback)
parser = argparse.ArgumentParser()
parser.add_argument("--env", type=str, default="dev", help="Deployment environment (dev/prod)")
parser.add_argument("--catalog", type=str, default="maven_market_uc", help="Target Unity Catalog")

# Detect if running interactively in a Databricks Notebook
if hasattr(sys, 'ps1') or 'IPython' in sys.modules:
    print("[INFO] Interactive Notebook environment detected. Using default dev arguments.")
    args, _ = parser.parse_known_args(args=["--env", "dev", "--catalog", "maven_market_uc"])
else:
    # Strict parsing when executed as a DABs spark_python_task
    args, _ = parser.parse_known_args()

# 2. Dynamically locate config.yml
def locate_config():
    for path_entry in [os.getcwd()] + sys.path:
        if not path_entry or not os.path.exists(path_entry):
            continue
        curr = os.path.abspath(path_entry)
        for _ in range(5):
            candidate = os.path.join(curr, "config.yml")
            if os.path.exists(candidate):
                return candidate
            parent = os.path.dirname(curr)
            if parent == curr:
                break
            curr = parent
    return None

config_path = locate_config()
if not config_path:
    raise FileNotFoundError("config.yml not found in the repository hierarchy.")

project_root = os.path.dirname(config_path)
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.custom_logger import CustomLogger

# 3. Initialize Spark Session & Logger
spark = SparkSession.builder.getOrCreate()

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

audit_table = config.get("tables", {}).get("audit_logs_table", f"{args.catalog}.audit.pipeline_audit_logs")
logger = CustomLogger(spark, f"kafka_synthetic_data_gen_{args.env}", target_table=audit_table)
logger.log_start()

try:
    kafka_config = config["sources"]["kafka"]
    bootstrap_servers = kafka_config["bootstrap_servers"]
    
    # 4. Fetch Kafka Secrets from Key Vault Scope
    kafka_key = dbutils.secrets.get(
        scope=kafka_config["secret_scope"], 
        key=kafka_config["api_key_name"]
    )
    kafka_secret = dbutils.secrets.get(
        scope=kafka_config["secret_scope"], 
        key=kafka_config["api_secret_name"]
    )

    # 5. Build SASL/JAAS Authentication Configuration String
    jaas_config = (
        f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
        f'username="{kafka_key}" password="{kafka_secret}";'
    )

    # 6. Synthesize Orders Data matching Schema (1000 records)
    orders_df = spark.sql("""
        SELECT 
            uuid() AS order_id,
            date_format(current_timestamp(), 'yyyy-MM-dd') AS transaction_date,
            date_format(date_sub(current_timestamp(), CAST(rand() * 10 AS INT)), 'yyyy-MM-dd') AS stock_date,
            CAST(rand() * 1559 + 1 AS INT) AS product_id,   
            CAST(rand() * 10280 + 1 AS INT) AS customer_id, 
            CAST(rand() * 500 + 1 AS INT) AS store_id,
            CAST(rand() * 10 + 1 AS INT) AS quantity,
            CAST(rand() * 100 AS DOUBLE) AS unit_price,
            cast(current_timestamp() as string) AS event_timestamp
        FROM explode(sequence(1, 1000))
    """)

    orders_kafka_df = orders_df.select(
        F.col("order_id").cast(StringType()).alias("key"),
        F.to_json(F.struct(
            "order_id", "transaction_date", "stock_date", 
            "product_id", "customer_id", "store_id", 
            "quantity", "unit_price", "event_timestamp"
        )).alias("value")
    )

    # 7. Synthesize Inventory Data matching Schema (500 records)
    inventory_df = spark.sql("""
        SELECT 
            uuid() AS event_id,
            CAST(rand() * 500 + 1 AS INT) AS store_id,
            CAST(rand() * 1559 + 1 AS INT) AS product_id,   
            CAST(rand() * 500 AS INT) AS stock_level,
            CAST(rand() * 50 + 10 AS INT) AS reorder_threshold,
            CAST(round(rand()) AS BOOLEAN) AS restock_flag,
            cast(current_timestamp() as string) AS event_timestamp
        FROM explode(sequence(1, 500))
    """)

    inventory_kafka_df = inventory_df.select(
        F.col("product_id").cast(StringType()).alias("key"),
        F.to_json(F.struct(
            "event_id", "store_id", "product_id", 
            "stock_level", "reorder_threshold", "restock_flag", 
            "event_timestamp"
        )).alias("value")
    )

    # 8. Write Data to Kafka
    def write_to_kafka(df, topic_name):
        (df.write
           .format("kafka")
           .option("kafka.bootstrap.servers", bootstrap_servers)
           .option("topic", topic_name)
           .option("kafka.security.protocol", "SASL_SSL")
           .option("kafka.sasl.mechanism", "PLAIN")
           .option("kafka.sasl.jaas.config", jaas_config)
           .save()
        )

    topic_orders = next(s["topic"] for s in kafka_config["streams"] if "orders" in s["topic"])
    topic_inventory = next(s["topic"] for s in kafka_config["streams"] if "inventory" in s["topic"])

    write_to_kafka(orders_kafka_df, topic_orders)
    write_to_kafka(inventory_kafka_df, topic_inventory)

    # 9. Record Metrics to Logs
    logger.log_success(records_processed=1500)
    print(f"Successfully published synthetic data to topics: {topic_orders} and {topic_inventory}.")

except Exception as e:
    logger.log_failure(e)
    raise e